<a href="https://colab.research.google.com/github/Moharram-Khaled/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Moharram-Khaled/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in the daily performance table represents one content page for one client on one reporting date. For the Ranking Signal Analysis lane, I can aggregate these daily observations into a monthly page-level view when the decision requires a monthly time window.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

files = list(api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
))

for file in files:
    print(file)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [12]:
from huggingface_hub import hf_hub_download
from google.colab import userdata
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

df = pd.read_parquet(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Shape: (9841378, 30)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Verification Query 1: Grain

key_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id"
]

total_rows = len(df)

unique_keys = df[key_columns].drop_duplicates().shape[0]

duplicate_rows = total_rows - unique_keys

print("Total rows:", total_rows)
print("Unique client + content + date keys:", unique_keys)
print("Duplicate rows:", duplicate_rows)

Total rows: 9841378
Unique client + content + date keys: 9841378
Duplicate rows: 0


In [14]:
# Verification Query 2: Row count and date span

print("Row count:", len(df))
print("Minimum report date:", df["report_date"].min())
print("Maximum report date:", df["report_date"].max())

Row count: 9841378
Minimum report date: 2026-03-01
Maximum report date: 2026-03-31


In [15]:
# Verification Query 3: GSC availability

import duckdb

query = """
SELECT
    COUNT(*) AS gsc_available_rows
FROM read_parquet(?)
WHERE gsc_data_available IS TRUE
"""

result = duckdb.sql(
    query,
    params=[file_path]
).df()

display(result)

,gsc_available_rows
0,3611061


In [16]:
features = [
    {
        "feature": "gsc_impressions",
        "available_when": "Known from the historical GSC observation window before the decision."
    },
    {
        "feature": "gsc_clicks",
        "available_when": "Known from the historical GSC observation window before the decision."
    },
    {
        "feature": "gsc_avg_position",
        "available_when": "Calculated from historical GSC position data before the decision."
    },
    {
        "feature": "ga4_sessions",
        "available_when": "Known from the historical GA4 observation window before the decision."
    },
    {
        "feature": "ga4_engaged_sessions",
        "available_when": "Known from historical GA4 engagement data before the decision."
    }
]

features_df = pd.DataFrame(features)

display(features_df)

,feature,available_when
0,gsc_impressions,Known from the historical GSC observation wind...
1,gsc_clicks,Known from the historical GSC observation wind...
2,gsc_avg_position,Calculated from historical GSC position data b...
3,ga4_sessions,Known from the historical GA4 observation wind...
4,ga4_engaged_sessions,Known from historical GA4 engagement data befo...


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

# Features

I will use five initial features for Ranking Signal Analysis:

* `gsc_impressions`
* `gsc_clicks`
* `gsc_avg_position`
* `ga4_sessions`
* `ga4_engaged_sessions`

These represent observable search, traffic, and engagement signals.

# Label

The outcome will be a future page-movement proxy derived from a later outcome window. I will not use future outcome information as a feature.

# Context

I will retain `report_date`, `client_hash_id`, `content_hash_id`, and the GSC/GA4 availability flags as context fields. These fields help identify the observation and interpret whether the relevant data source was available.

# Excluded

I will exclude future-derived or label-derived fields from the feature set because they would not be available at the decision moment and could introduce target leakage. I will also exclude fields that are not needed for the initial Ranking Signal Analysis feature frame.


* `gsc_impressions` — available before the decision because it comes from the historical GSC observation window.
* `gsc_clicks` — available before the decision because historical clicks are recorded in GSC.
* `gsc_avg_position` — available before the decision because it is calculated from historical GSC position data.
* `ga4_sessions` — available before the decision because historical sessions are recorded in GA4.
* `ga4_engaged_sessions` — available before the decision because historical engagement sessions are recorded in GA4.


In [17]:
feature_frame = df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions"
    ]
].copy()

display(feature_frame.head(10))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,NaN
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,NaN,NaN
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,NaN,NaN
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,NaN,NaN
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,NaN,NaN
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,NaN,NaN


### Leakage check

I would treat any future movement field or field derived from the future outcome as label-derived. Including such a field would leak information from the outcome into the features and make the model score misleadingly high. These fields remain excluded from the honest feature set.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification results

The March 2026 slice contains 9,841,378 daily page-client observations from 2026-03-01 through 2026-03-31. The grain check found no duplicate combinations of report date, client, and content. GSC data is available for 3,611,061 observations. The five selected features are based on historical data from the decision window.



In [19]:
# Section 3: Verification checks

print("=== Grain ===")
print("Total rows:", len(df))

unique_keys = df[
    ["report_date", "client_hash_id", "content_hash_id"]
].drop_duplicates().shape[0]

print("Unique keys:", unique_keys)
print("Duplicate rows:", len(df) - unique_keys)

print("\n=== Date window ===")
print("Minimum date:", df["report_date"].min())
print("Maximum date:", df["report_date"].max())

print("\n=== GSC availability ===")

availability_query = """
SELECT COUNT(*) AS gsc_available_rows
FROM read_parquet(?)
WHERE gsc_data_available IS TRUE
"""

availability_result = duckdb.sql(
    availability_query,
    params=[file_path]
).df()

display(availability_result)

print("\n=== Feature missingness ===")

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

missingness = df[feature_columns].isna().sum().to_frame("missing_rows")

display(missingness)

=== Grain ===
Total rows: 9841378
Unique keys: 9841378
Duplicate rows: 0

=== Date window ===
Minimum date: 2026-03-01
Maximum date: 2026-03-31

=== GSC availability ===


,gsc_available_rows
0,3611061



=== Feature missingness ===


,missing_rows
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,6230317
ga4_sessions,3018741
ga4_engaged_sessions,3018741


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice is limited to March 2026 and therefore may not represent behavior across all months. The daily performance table also contains different levels of GSC and GA4 availability, so some observations have incomplete coverage from one or both sources. In addition, the daily observations can be aggregated into monthly features, but overlapping time windows must be handled carefully to avoid using information from the outcome period.


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.